# STAI-X 2026 — Feature Engineering Validation

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_train, load_val
from src.features import FeaturePipeline, CENSUS_DIVISION

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

In [ ]:
train = load_train()
val   = load_val()
print('Train:', train.shape, '  Val:', val.shape)

## Run pipeline

In [ ]:
pipeline = FeaturePipeline(n_svd=20)
train_feat, val_feat = pipeline.fit_transform(train, val)
print('Feature count:', len(pipeline.feature_cols))
print('Features:', pipeline.feature_cols)

## Sanity: no NaN in engineered features

In [ ]:
nan_train = train_feat[pipeline.feature_cols].isna().sum().sum()
nan_val   = val_feat[pipeline.feature_cols].isna().sum().sum()
print(f'NaN in train features: {nan_train}  (expected 0)')
print(f'NaN in val features:   {nan_val}  (expected 0)')

## Census division coverage

In [ ]:
missing_div = set(train['jurisdiction'].unique()) - set(CENSUS_DIVISION.keys())
print('Jurisdictions missing census division mapping:', missing_div or 'None')
train_feat['census_division'].value_counts().sort_index().plot(kind='bar', title='Census division dist')
plt.tight_layout()
plt.show()

## TF-IDF SVD components — variance explained

In [ ]:
explained = pipeline.svd.explained_variance_ratio_
plt.figure(figsize=(8, 3))
plt.bar(range(len(explained)), explained)
plt.xlabel('SVD component')
plt.ylabel('Variance explained')
plt.title('TF-IDF SVD explained variance')
plt.tight_layout()
plt.show()
print(f'Cumulative variance (all {len(explained)} components): {explained.sum():.3f}')

## Period rank distribution

In [ ]:
print('Train period_rank unique values:', sorted(train_feat['period_rank'].unique()))
print('Val   period_rank unique values:', sorted(val_feat['period_rank'].unique()))
print('\nLag feature fill rate in train:')
for col in ['rate_lag_1', 'rate_lag_2']:
    if col in train_feat.columns:
        pct = train_feat[col].notna().mean()
        print(f'  {col}: {pct:.1%} non-null (before imputation)')

## Feature correlation heatmap (numeric covariates)

In [ ]:
base_cols = ['unemployment_rate','labor_force','temp_avg_f','precip_in',
             'gtrends_overdose','gtrends_fentanyl','gtrends_naloxone',
             'gtrends_opioid','gtrends_methamphetamine',
             'census_division','log_labor_force','doh_has_release',
             'gtrends_fentanyl_x_opioid','gtrends_overdose_x_naloxone']
base_cols = [c for c in base_cols if c in train_feat.columns]
corr_m = train_feat[base_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_m, cmap='RdBu_r', center=0, square=True, linewidths=0.3, annot=False)
plt.title('Feature correlation matrix')
plt.tight_layout()
plt.show()